# AIC 2026 - Object descriptions on Colab

Thin launcher only. Select a GPU runtime, store a read-only GitHub token in Colab Secrets as `AIC_GITHUB_TOKEN`, and change the documented paths below. Never paste tokens into the notebook.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess

drive.mount("/content/drive")
target = Path("/content/AIC-2026")
if not target.is_dir():
    token = userdata.get("AIC_GITHUB_TOKEN")
    clone_env = os.environ.copy()
    clone_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}"})
    subprocess.run(["git", "clone", "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git", str(target)], env=clone_env, check=True)
    del token, clone_env
os.chdir(target)
print(Path.cwd())

Replace the video ID and mounted data paths. `%env` persists them for later `%%bash` cells.

In [ ]:
%env AIC_DATA_ROOT=/content/drive/MyDrive/AIC2026/data
%env AIC_ARTIFACT_ROOT=/content/drive/MyDrive/AIC2026/artifacts
%env AIC_CACHE_ROOT=/content/aic2026-cache
%env AIC_VIDEO_ID=replace_with_video_id

In [ ]:
%%bash
set -euo pipefail
python -m pip install --no-deps -r requirements/object-description.txt
python scripts/verify_environment.py \
  --config configs/offline/object_description.yaml \
  --device cuda --write-report

Smoke run on two frames uses an isolated `smoke/` artifact tree. For a full run, use a different output tree and remove `--limit 2`; never resume a limited artifact as a full run.

In [ ]:
%%bash
set -euo pipefail
SMOKE_ROOT="$AIC_ARTIFACT_ROOT/smoke"
FRAME_MANIFEST="$SMOKE_ROOT/frame_manifests/$AIC_VIDEO_ID.jsonl"
MASK_ARTIFACT="$SMOKE_ROOT/object_description/masks/$AIC_VIDEO_ID.jsonl"
DESCRIPTION_ARTIFACT="$SMOKE_ROOT/object_description/descriptions/$AIC_VIDEO_ID.jsonl"
MAP_CSV="$AIC_DATA_ROOT/map-keyframes/$AIC_VIDEO_ID.csv"
FRAMES_DIR="$AIC_DATA_ROOT/keyframes/$AIC_VIDEO_ID"
OBJECTS_DIR="$AIC_DATA_ROOT/objects/$AIC_VIDEO_ID"
python scripts/build_frame_manifest.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --map-csv "$MAP_CSV" --frames-dir "$FRAMES_DIR" --output "$FRAME_MANIFEST" --resume --limit 2
python scripts/prepare_object_masks.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --frame-manifest "$FRAME_MANIFEST" --objects-dir "$OBJECTS_DIR" --output "$MASK_ARTIFACT" --device cuda --resume --limit 2
python scripts/run_dam_descriptions.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --mask-artifact "$MASK_ARTIFACT" --output "$DESCRIPTION_ARTIFACT" --device cuda --resume --limit 2
python scripts/validate_artifacts.py --artifact "$DESCRIPTION_ARTIFACT" --manifest "${DESCRIPTION_ARTIFACT%.jsonl}.manifest.json" --require-captions